# SSTW-v2 G0 Flow-guidance primitive

This `DIAGNOSTIC_ONLY` run answers one question: can a fixed two-coordinate observer provide two independent, directionally correct, quality-bounded control axes on the real Wan Flow state? It does not write MP4 or run AISB/calibration/Viterbi.

A100 80GB is recommended because this diagnostic differentiates through the Wan VAE. L4 is allowed, but an OOM is `INSTRUMENTATION_INSUFFICIENT`, not a method result.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, os, pathlib, shutil, subprocess, sys, zipfile, hashlib

REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW-Feasibility.git'
AUTHORIZED_REF = '500171b7aed06787532f5aba5db36152a075f58d'
RUN_ID = '78a8b69a7a255c9b'
AUTHORIZE_EXECUTION = True
AUTHORIZE_DRIVE_IO = True
WORK = pathlib.Path('/content/sstw-v2-g0-work')
RUN_ROOT = pathlib.Path(f'/content/sstw-v2-g0-{RUN_ID}')
OUTPUT = RUN_ROOT / 'output'
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/SC-SSTW-Feasibility/sstw-v2-g0-flow-guidance-unipc-scalar-fix')
DRIVE_ARCHIVE = DRIVE_ROOT / f'sstw-v2-g0-{RUN_ID}.zip'
DRIVE_SIDECAR = DRIVE_ROOT / f'sstw-v2-g0-{RUN_ID}.zip.sha256.json'
assert AUTHORIZE_EXECUTION and AUTHORIZE_DRIVE_IO
import torch
runtime = {'torch': torch.__version__, 'cuda': torch.version.cuda, 'cuda_available': torch.cuda.is_available(), 'bf16_supported': bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported()), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'vram_gib': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2) if torch.cuda.is_available() else None}
print('Runtime diagnostic:', runtime)
if not runtime['cuda_available'] or not runtime['bf16_supported']:
    raise RuntimeError('CUDA and BF16 capabilities are required')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'diffusers==0.35.2', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub'])
if WORK.exists(): shutil.rmtree(WORK)
if RUN_ROOT.exists(): shutil.rmtree(RUN_ROOT)
subprocess.check_call(['git', 'clone', '--filter=blob:none', REPOSITORY_URL, str(WORK)])
subprocess.check_call(['git', '-C', str(WORK), 'checkout', '--detach', AUTHORIZED_REF])
resolved = subprocess.check_output(['git', '-C', str(WORK), 'rev-parse', 'HEAD'], text=True).strip()
if not resolved.startswith(AUTHORIZED_REF): raise RuntimeError('authorized implementation ref mismatch')
RUN_ROOT.mkdir(parents=True)
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print({'authorized_commit': resolved, 'run_id': RUN_ID, 'drive_archive': str(DRIVE_ARCHIVE)})

In [ ]:
stdout_path = RUN_ROOT / 'runner.stdout.txt'
stderr_path = RUN_ROOT / 'runner.stderr.txt'
argv = [sys.executable, str(WORK / 'experiments/run_g0_flow_guidance_primitive.py'), '--output', str(OUTPUT)]
with stdout_path.open('w') as out, stderr_path.open('w') as err:
    completed = subprocess.run(argv, cwd=WORK, stdout=out, stderr=err)
stdout = stdout_path.read_text()
stderr = stderr_path.read_text()
print('Runner return code:', completed.returncode)
print('Runner stdout:\n' + stdout)
if stderr: print('Runner stderr:\n' + stderr)
report = json.loads([line for line in stdout.splitlines() if line.strip()][-1])
if completed.returncode not in (0, 2, 3): raise RuntimeError('unexpected runner return code')
if report.get('status') not in ('FLOW_GUIDANCE_PRIMITIVE_FEASIBLE', 'FLOW_GUIDANCE_PRIMITIVE_NOT_FEASIBLE', 'INSTRUMENTATION_INSUFFICIENT'): raise RuntimeError('unexpected G0 status')
local_archive = pathlib.Path(f'/content/sstw-v2-g0-{RUN_ID}.zip')
with zipfile.ZipFile(local_archive, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_ROOT.rglob('*')):
        if path.is_file(): archive.write(path, path.relative_to(RUN_ROOT.parent))
archive_sha256 = hashlib.sha256(local_archive.read_bytes()).hexdigest()
shutil.copy2(local_archive, DRIVE_ARCHIVE)
sidecar = {'schema_version': 1, 'diagnostic_class': 'DIAGNOSTIC_ONLY', 'run_id': RUN_ID, 'authorized_ref': resolved, 'archive_sha256': archive_sha256, 'status': report['status']}
DRIVE_SIDECAR.write_text(json.dumps(sidecar, sort_keys=True) + '\n')
if hashlib.sha256(DRIVE_ARCHIVE.read_bytes()).hexdigest() != archive_sha256: raise RuntimeError('Drive ZIP copy failed')
summary = {**report, 'run_id': RUN_ID, 'archive_sha256': archive_sha256, 'drive_zip': str(DRIVE_ARCHIVE), 'drive_sidecar': str(DRIVE_SIDECAR)}
print(summary)
if report['status'] == 'INSTRUMENTATION_INSUFFICIENT': raise RuntimeError('G0 instrumentation insufficient; see packaged runner logs')